In [1]:
import sys
import pandas as pd
from pathlib import Path
from linearmodels.panel import PanelOLS
sys.path.append(str(Path.cwd().parent))
from constants import PATH_TO_FINAL_OUTPUT
import statsmodels.api as sm


In [10]:
df = pd.read_csv(PATH_TO_FINAL_OUTPUT)

In [11]:
df['debate_date'] = pd.to_datetime(df['debate_date'])
df['speaker_first_debate_date'] = pd.to_datetime(df['speaker_first_debate_date'])

motion_balance = df[df['side'] == 'aff'].groupby(['debate_id', 'motion'])['ballots_gained'].mean().reset_index()
motion_balance = motion_balance.groupby('motion')['ballots_gained'].mean().reset_index()
motion_balance.columns = ['motion', 'motion_balance']

df = df.merge(motion_balance, on='motion', how='left')

df['years_since_first_debate'] = df['debate_date'].dt.year - df['speaker_first_debate_date'].dt.year

df['tournament_round'] = df.groupby('tournament_id')['debate_date'].rank(method='dense').astype(int)

df['is_aff'] = (df['side'] == 'aff').astype(int)
df['motion_balance_x_aff'] = df['motion_balance'] * df['is_aff']

In [12]:
def get_teammate_avg(group):
    teammate_avgs = []
    for idx in group.index:
        other_scores = group.loc[group.index != idx, 'speaker_points']
        teammate_avgs.append(other_scores.mean())
    return pd.Series(teammate_avgs, index=group.index)

df['avg_teammate_score'] = df.groupby(['debate_id', 'side'], group_keys=False).apply(get_teammate_avg)

In [13]:
df_reg = df.dropna(subset=['speaker_points', 'is_male', 'years_since_first_debate', 
                            'tournament_round', 'motion_balance_x_aff', 'motion_balance', 
                            'speaker_name', 'avg_teammate_score'])

X_pooled = df_reg[['is_male', 'years_since_first_debate', 'tournament_round', 
                    'motion_balance_x_aff', 'motion_balance', 'avg_teammate_score']].astype(float)
y_pooled = df_reg['speaker_points'].astype(float)

X_pooled = sm.add_constant(X_pooled)
model_pooled = sm.OLS(y_pooled, X_pooled).fit()

In [14]:
df_panel = df_reg.copy()
df_panel['speaker_id'] = pd.Categorical(df_panel['speaker_name']).codes
df_panel = df_panel.set_index(['speaker_id', 'debate_date'])

y_panel = df_panel['speaker_points']
# we have to drop years since first debate and is_male to use the fixed effects model
X_panel = df_panel[['tournament_round', 
                     'motion_balance_x_aff', 'motion_balance', 'avg_teammate_score']].astype(float)


model_fixed_effects = PanelOLS(y_panel, X_panel, entity_effects=True).fit()

In [15]:
print("="*80)
print("Pooled OLS Regression Results")
print("="*80)
print(model_pooled.summary())

print("\n" + "="*80)
print("Fixed Effects Panel Regression Results (Speaker Fixed Effects)")
print("="*80)
print(model_fixed_effects.summary)


print("\n" + "="*80)
print("Model Comparison")
print("="*80)
print(f"Pooled OLS R-squared: {model_pooled.rsquared:.4f}")
print(f"Fixed Effects R-squared: {model_fixed_effects.rsquared:.4f}")
print(f"Fixed Effects R-squared (within): {model_fixed_effects.rsquared_within:.4f}")

Pooled OLS Regression Results
                            OLS Regression Results                            
Dep. Variable:         speaker_points   R-squared:                       0.586
Model:                            OLS   Adj. R-squared:                  0.582
Method:                 Least Squares   F-statistic:                     155.8
Date:                Thu, 22 Jan 2026   Prob (F-statistic):          6.62e-123
Time:                        20:53:19   Log-Likelihood:                -1695.9
No. Observations:                 667   AIC:                             3406.
Df Residuals:                     660   BIC:                             3437.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------